In [4]:
import pandas as pd

In [5]:
reports_df = pd.read_csv('../chest-x-ray-data/indiana_reports.csv')
projections_df = pd.read_csv('../chest-x-ray-data/indiana_projections.csv')

In [6]:
expected_uids = set(range(1, 4000))
actual_uids = set(reports_df['uid'])

# Identify missing UIDs
missing_uids = sorted(list(expected_uids - actual_uids))

# Identify repeated UIDs
uid_counts = reports_df['uid'].value_counts()
repeated_uids = uid_counts[uid_counts > 1].index.tolist()

print(f"Missing UIDs (Total: {len(missing_uids)}): {missing_uids}")
print(f"Repeated UIDs (Total: {len(repeated_uids)}): {repeated_uids}")

Missing UIDs (Total: 148): [109, 140, 148, 156, 180, 199, 213, 231, 232, 265, 311, 369, 381, 447, 528, 566, 574, 625, 678, 724, 775, 789, 807, 816, 823, 848, 852, 873, 876, 895, 898, 924, 1068, 1080, 1127, 1132, 1181, 1185, 1186, 1201, 1215, 1227, 1247, 1251, 1298, 1299, 1325, 1486, 1490, 1495, 1507, 1611, 1613, 1628, 1692, 1741, 1749, 1772, 1858, 1862, 1869, 1890, 1917, 1955, 1989, 1996, 1998, 2004, 2009, 2037, 2051, 2057, 2064, 2076, 2101, 2104, 2107, 2182, 2196, 2284, 2309, 2346, 2399, 2429, 2452, 2500, 2508, 2521, 2534, 2556, 2598, 2602, 2603, 2641, 2675, 2678, 2682, 2703, 2707, 2732, 2736, 2757, 2800, 2823, 2834, 2846, 2849, 2853, 2869, 2872, 2875, 2883, 2896, 2900, 2907, 2912, 2913, 2914, 2973, 2987, 2990, 3007, 3035, 3055, 3107, 3161, 3276, 3293, 3295, 3308, 3351, 3397, 3425, 3447, 3463, 3476, 3484, 3554, 3558, 3622, 3710, 3711, 3788, 3800, 3859, 3864, 3920, 3990]
Repeated UIDs (Total: 0): []


In [7]:
# 1. Keep only Frontal projections
projections_df = projections_df[projections_df['projection'] == 'Frontal']

# 2. Deduplicate UIDs (keep first occurrence for the 122 cases with multiple frontals)
projections_df = projections_df.drop_duplicates(subset=['uid'], keep='first')

print(f"Final count of unique frontal images for linking: {len(projections_df)}")

Final count of unique frontal images for linking: 3689


In [8]:
reports_df.head(5)
# projections_df.head(5)

,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [9]:
# add images in projection_df to reports_df based on uid
reports_df = reports_df.merge(projections_df, on='uid',how='right') 

In [10]:
reports_df.head(5)
reports_df = reports_df.drop(columns=['projection'])

In [11]:
reports_df.to_csv('../chest-x-ray-data/indiana_reports_with_projections.csv', index=False)